In [26]:
import optuna
import numpy as np

In [27]:
# ============================
# B-сплайновый базис (Кокс-де Бор)
# ============================

def make_knot_vector(a, b, n_internal, degree=3):
    """Расширенный вектор узлов (clamped)."""
    internal = np.linspace(a, b, n_internal + 2)
    return np.concatenate([
        np.repeat(a, degree), internal, np.repeat(b, degree)
    ])


def bspline_basis(x, knots, degree=3):
    """Матрица B-сплайнового базиса (N x K)."""
    x = np.asarray(x, dtype=float)
    knots = np.asarray(knots, dtype=float)
    N = len(x)

    B = np.zeros((N, len(knots) - 1))
    for i in range(len(knots) - 1):
        if knots[i] < knots[i + 1]:
            B[(x >= knots[i]) & (x < knots[i + 1]), i] = 1.0
    for i in range(len(knots) - 2, -1, -1):
        if knots[i] < knots[i + 1]:
            B[x == knots[-1], i] = 1.0
            break

    for d in range(1, degree + 1):
        B_new = np.zeros((N, len(knots) - 1 - d))
        for i in range(len(knots) - 1 - d):
            d1 = knots[i + d] - knots[i]
            d2 = knots[i + d + 1] - knots[i + 1]
            if d1 > 0:
                B_new[:, i] += (x - knots[i]) / d1 * B[:, i]
            if d2 > 0:
                B_new[:, i] += (knots[i + d + 1] - x) / d2 * B[:, i + 1]
        B = B_new
    return B


def diff_matrix(K, order=2):
    """Матрица конечных разностей порядка order."""
    D = np.eye(K)
    for _ in range(order):
        D = D[1:, :] - D[:-1, :]
    return D


# ============================
# 2D тензорный P-сплайн
# ============================

def pspline_2d_fit(x, y, Z, n_basis=30, degree=3, penalty_order=2, lam=1.0, x_range=None, y_range=None):
    """
    Подгонка 2D P-сплайна (тензорное произведение).
    Возвращает коэффициенты A и узловые вектора kx, ky.
    """
    if x_range is None:
        x_range = (x[0], x[-1])
    if y_range is None:
        y_range = (y[0], y[-1])

    kx = make_knot_vector(x_range[0], x_range[1], n_basis, degree)
    ky = make_knot_vector(y_range[0], y_range[1], n_basis, degree)
    Bx = bspline_basis(x, kx, degree)
    By = bspline_basis(y, ky, degree)

    Kx, Ky = Bx.shape[1], By.shape[1]

    Dx = diff_matrix(Kx, penalty_order)
    Dy = diff_matrix(Ky, penalty_order)

    BxTBx = Bx.T @ Bx
    ByTBy = By.T @ By
    DxTDx = Dx.T @ Dx
    DyTDy = Dy.T @ Dy

    rhs = (Bx.T @ Z @ By).ravel()

    LHS = (np.kron(ByTBy, BxTBx)
           + lam * np.kron(np.eye(Ky), DxTDx)
           + lam * np.kron(DyTDy, np.eye(Kx)))

    alpha = np.linalg.solve(LHS, rhs)
    A = alpha.reshape(Kx, Ky)

    return A, kx, ky


def pspline_2d_eval(A, kx, ky, x_eval, y_eval, degree=3):
    """Вычисление P-сплайна на сетке."""
    Bx = bspline_basis(x_eval, kx, degree)
    By = bspline_basis(y_eval, ky, degree)
    return Bx @ A @ By.T

In [28]:
Z_raw = np.load("./data/fuji_elevation.npy").astype(np.float64)
lat = np.load("./data/fuji_lat.npy")
lon = np.load("./data/fuji_lon.npy")

Z_raw = Z_raw[::-1, :]
lat_sn = lat[::-1]

X_coords = lon
Y_coords = lat_sn

step_points = 8

X_pts = X_coords[::step_points]
Y_pts = Y_coords[::step_points]
Z_pts = Z_raw[::step_points, ::step_points]

# n_basis = 40

In [29]:
def objective(trial):
    lam = trial.suggest_float("lam", 0.01, 1)
    n_basis = trial.suggest_int("n_basis", 30, 100)

    A, kx, ky = pspline_2d_fit(
        X_pts, Y_pts, Z_pts.T,   # тоже транспонируем
        n_basis=n_basis, degree=3, penalty_order=2, lam=lam,
        x_range=(X_coords[0], X_coords[-1]),
        y_range=(Y_coords[0], Y_coords[-1])
    )
    Z_interp_on_raw = pspline_2d_eval(A, kx, ky, X_coords, Y_coords, degree=3)
    mae = np.nanmean(np.abs(Z_interp_on_raw - Z_raw.T))
    return mae

In [30]:
study = optuna.create_study(direction="minimize")  # решение на минимум
study.optimize(objective, n_trials=100)

study.best_params

[I 2026-05-11 22:40:17,912] A new study created in memory with name: no-name-275a26da-8c56-4398-9923-dddbcbe70992
[I 2026-05-11 22:40:18,781] Trial 0 finished with value: 12.678912974568266 and parameters: {'lam': 0.26179584191142413, 'n_basis': 40}. Best is trial 0 with value: 12.678912974568266.
[I 2026-05-11 22:40:20,814] Trial 1 finished with value: 9.394788755408946 and parameters: {'lam': 0.13040061111830936, 'n_basis': 62}. Best is trial 1 with value: 9.394788755408946.
[I 2026-05-11 22:40:21,554] Trial 2 finished with value: 12.23076078313646 and parameters: {'lam': 0.5237945995976058, 'n_basis': 52}. Best is trial 1 with value: 9.394788755408946.
[I 2026-05-11 22:40:22,191] Trial 3 finished with value: 12.426030568368875 and parameters: {'lam': 0.6653302222883234, 'n_basis': 54}. Best is trial 1 with value: 9.394788755408946.
[I 2026-05-11 22:40:22,946] Trial 4 finished with value: 12.738583115051577 and parameters: {'lam': 0.7456188367135388, 'n_basis': 53}. Best is trial 1 w

{'lam': 0.012254341711072526, 'n_basis': 89}

In [31]:
Z_raw = np.load("./data/fuji_elevation.npy").astype(np.float64)
lat = np.load("./data/fuji_lat.npy")
lon = np.load("./data/fuji_lon.npy")

Z_raw = Z_raw[::-1, :]
lat_sn = lat[::-1]

X_coords = lon
Y_coords = lat_sn

step_points = 8

X_pts = X_coords[::step_points]
Y_pts = Y_coords[::step_points]
Z_pts = Z_raw[::step_points, ::step_points]

np.random.seed(42)
noise_std = 30.0
Z_pts_noisy = Z_pts + np.random.normal(0, noise_std, Z_pts.shape)

n_basis = 40

# делим строки опорной сетки на train/val через одну
train_idx = np.arange(0, len(Y_pts), 2)
val_idx   = np.arange(1, len(Y_pts), 2)

Y_train = Y_pts[train_idx]
Z_train = Z_pts_noisy[train_idx, :]

Y_val = Y_pts[val_idx]
Z_val = Z_pts_noisy[val_idx, :]

train_col_idx = np.arange(0, len(X_pts), 2)
val_col_idx   = np.arange(1, len(X_pts), 2)

X_train = X_pts[train_col_idx]
X_val   = X_pts[val_col_idx]

Z_train = Z_pts_noisy[:, train_col_idx]  # (Ny, Nx_train)
Z_val   = Z_pts_noisy[:, val_col_idx]    # (Ny, Nx_val)

In [32]:
def objective_noisy(trial):
    lam = trial.suggest_float("lam", 1e-3, 10.0, log=True)
    n_basis = trial.suggest_int("n_basis", 30, 100)

    A, kx, ky = pspline_2d_fit(
        X_train, Y_pts, Z_train.T,   # (Ny, Nx_train) -> (Nx_train, Ny)
        n_basis=n_basis, degree=3, penalty_order=2, lam=lam,
        x_range=(X_coords[0], X_coords[-1]),
        y_range=(Y_coords[0], Y_coords[-1])
    )

    # pspline_2d_eval вернёт (Nx_val, Ny), val лежит как (Ny, Nx_val)
    Z_pred_val = pspline_2d_eval(A, kx, ky, X_val, Y_pts, degree=3)
    mae = np.nanmean(np.abs(Z_pred_val - Z_val.T))
    return mae

In [33]:
study_noisy = optuna.create_study(direction="minimize")
study_noisy.optimize(objective_noisy, n_trials=100)

print(study_noisy.best_params)

[I 2026-05-11 22:47:41,936] A new study created in memory with name: no-name-4fb7c783-2c9f-4fd8-9f52-5fc125373a58


[I 2026-05-11 22:47:50,865] Trial 0 finished with value: 36.551871668626994 and parameters: {'lam': 4.372577940202084, 'n_basis': 85}. Best is trial 0 with value: 36.551871668626994.
[I 2026-05-11 22:47:58,871] Trial 1 finished with value: 55.62034584130248 and parameters: {'lam': 0.017963514458059257, 'n_basis': 87}. Best is trial 0 with value: 36.551871668626994.
[I 2026-05-11 22:48:06,538] Trial 2 finished with value: 38.08567457303552 and parameters: {'lam': 0.07992904375246919, 'n_basis': 87}. Best is trial 0 with value: 36.551871668626994.
[I 2026-05-11 22:48:07,188] Trial 3 finished with value: 88.41484562892477 and parameters: {'lam': 0.0011949119780250168, 'n_basis': 50}. Best is trial 0 with value: 36.551871668626994.
[I 2026-05-11 22:48:08,143] Trial 4 finished with value: 38.99107657468712 and parameters: {'lam': 6.595484499116474, 'n_basis': 56}. Best is trial 0 with value: 36.551871668626994.
[I 2026-05-11 22:48:18,743] Trial 5 finished with value: 65.45377359867197 and p

{'lam': 0.1320515764016064, 'n_basis': 56}
